In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.conf import SparkConf
from pyspark.context import SparkContext

In [2]:
credentials_location = '/home/harsh/Data-engineering-zoomcamp/batch_processing_spark/kestra-sandbox-499212-740514085025.json'
conf = SparkConf() \
    .setMaster('local[*]') \
    .setAppName('test') \
    .set("spark.jars", "./lib/gcs-connector-hadoop3-2.2.25-shaded.jar") \
    .set("spark.hadoop.google.cloud.auth.service.account.enable", "true") \
    .set("spark.hadoop.google.cloud.auth.service.account.json.keyfile", credentials_location)


'''
 Root cause: PySpark 4.2.0 bundles Hadoop 3.5.0, and Hadoop 3.5's core-default.xml now ships built-in defaults for fs.gs.* keys (e.g. fs.gs.block.size = 64m) using human-readable size suffixes. The third-party GCS connector jar in lib/ parses these as plain integers, so it threw NumberFormatException: For input string: "64m" while initializing the gs:// filesystem — this happened regardless of which connector jar was selected (tried both gcs-connector-3.0.9-shaded.jar and gcs-connector-hadoop3-2.2.25-shaded.jar).
2. Fix: explicitly overrode the affected keys (fs.gs.block.size, fs.gs.rewrite.max.chunk.size, fs.gs.outputstream.buffer.size, fs.gs.inputstream.inplace.seek.limit, fs.gs.inputstream.min.range.request.size) with plain byte-count values on hadoop_conf after the GCS filesystem impl is registered.'''

In [3]:
sc = SparkContext(conf=conf)

hadoop_conf = sc._jsc.hadoopConfiguration()

hadoop_conf.set("fs.AbstractFileSystem.gs.impl",  "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFS")
hadoop_conf.set("fs.gs.impl", "com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystem")
hadoop_conf.set("fs.gs.auth.service.account.json.keyfile", credentials_location)
hadoop_conf.set("fs.gs.auth.service.account.enable", "true")
hadoop_conf.set("fs.gs.block.size", "67108864")
hadoop_conf.set("fs.gs.rewrite.max.chunk.size", "536870912")
hadoop_conf.set("fs.gs.outputstream.buffer.size", "8388608")
hadoop_conf.set("fs.gs.inputstream.inplace.seek.limit", "8388608")
hadoop_conf.set("fs.gs.inputstream.min.range.request.size", "2097152")

26/08/14 14:59:46 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/harsh/Data-engineering-zoomcamp/batch_processing_spark/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [4]:
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

In [5]:
spark.sparkContext.setLogLevel("WARN")

In [6]:
df_green = spark.read.parquet('gs://spark-tutorial-pq/pq/green/*/*')

26/08/14 14:59:59 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: gs://spark-tutorial-pq/pq/green/*/*.
java.io.FileNotFoundException: File not found: gs://spark-tutorial-pq/pq/green/*/*
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.lambda$getFileStatus$10(GoogleHadoopFileSystemBase.java:1088)
	at com.google.cloud.hadoop.fs.gcs.GhfsStorageStatistics.trackDuration(GhfsStorageStatistics.java:116)
	at com.google.cloud.hadoop.fs.gcs.GoogleHadoopFileSystemBase.getFileStatus(GoogleHadoopFileSystemBase.java:1073)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:394)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:210)
	at org.apache.spark.sql.catalyst.analysis.Resol

In [7]:
df_green.count()

2304517

In [8]:
df_green.show()

+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|VendorID|lpep_pickup_datetime|lpep_dropoff_datetime|store_and_fwd_flag|RatecodeID|PULocationID|DOLocationID|passenger_count|trip_distance|fare_amount|extra|mta_tax|tip_amount|tolls_amount|ehail_fee|improvement_surcharge|total_amount|payment_type|trip_type|congestion_surcharge|
+--------+--------------------+---------------------+------------------+----------+------------+------------+---------------+-------------+-----------+-----+-------+----------+------------+---------+---------------------+------------+------------+---------+--------------------+
|       2| 2020-01-23 13:10:15|  2020-01-23 13:38:16|                 N|         1|          74|         130|              1|        12.77|       36.0|  0.0|    0.

In [10]:
spark.stop()